# Dataset Aggregation and Preprocessing

We have gatherd our data by means of the RAWG API. The API calls give us individual csv files that are all stored in the `detailed_games_by_year` folder. We will merge these into a single DataFrame, apply preprocessing to clean the dataset,
and returns a structured DataFrame ready for use in our visualisations.

In [1]:
import pandas as pd
import os
import glob
import ast

# Path to the folder containing the csv files
folder_path = './api data/detailed_games_by_year'

In [2]:
# Collect all CSV files from the folder
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Load and concatenate all CSVs into a single DataFrame
df = pd.concat(
    (pd.read_csv(file, low_memory=False) for file in csv_files),
    ignore_index=True
)

## Preprocessing

1. Some columns contain JSON-like structures (e.g., platforms, tags, genres). We simplify these by flattening them.

In [4]:
# Flatten nested columns
def extract_names(entry):
    if pd.isna(entry):
        return None
    try:
        data = ast.literal_eval(entry) if isinstance(entry, str) else entry
        if isinstance(data, list):
            return ", ".join(d.get('name', str(d)) for d in data if isinstance(d, dict))
    except Exception:
        return str(entry)
    return str(entry)

nested_cols = ['platforms', 'parent_platforms', 'tags', 'genres', 'developers', 'publishers', 'stores']
for col in nested_cols:
    if col in df.columns:
        df[col] = df[col].apply(extract_names)

2. We remove columns that are mostly empty to keep only useful information. Specifically, we drop any column with more than 95% missing values.

In [ ]:
# Drop columns with >90% missing values
# missing_ratio = all_files_df.isna().mean()
# columns_to_keep = missing_ratio[missing_ratio < 0.9].index
# columns_dropped = missing_ratio[missing_ratio >= 0.9].index.tolist()
# print(f"Dropping {len(columns_dropped)} columns with >90% missing values:\n", columns_dropped)
# df = all_files_df[columns_to_keep]

3. Ensure key columns have the correct data types for filtering and plotting

In [5]:
# Convert types
df['released'] = pd.to_datetime(df['released'], errors='coerce')
for col in ['metacritic', 'rating', 'rating_top', 'playtime', 'added']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

4. Extract Year from file title

In [6]:
# Extract release year
df['release_year'] = df['released'].dt.year

5. Drop columns that contain nearly the same value for all rows.

In [7]:
# Drop low-variance columns
low_variance = [col for col in df.columns if df[col].nunique(dropna=True) <= 1]
print("Dropping low-variance columns:", low_variance)
df.drop(columns=low_variance, inplace=True)

Dropping low-variance columns: ['user_game', 'saturated_color', 'dominant_color', 'esrb_rating', 'clip', 'community_rating', 'reactions', 'added_by_status', 'redirect']


6. Expand ratings into individual columns (e.g., ratings_exceptional, ratings_meh, etc.)

In [15]:
def parse_rating_summary(entry):
    if pd.isna(entry): return {}
    try:
        data = ast.literal_eval(entry)
        return {d.get('title', 'unknown'): d.get('count', 0) for d in data if isinstance(d, dict)}
    except:
        return {}

# Apply only if 'ratings' is in the dataframe
if 'ratings' in df.columns:
    rating_data = df['ratings'].apply(parse_rating_summary)
    rating_df = pd.DataFrame(rating_data.tolist()).fillna(0).astype(int)
    df = pd.concat([df.drop(columns=['ratings']), rating_df.add_prefix("ratings_")], axis=1)

## Final Output

The DataFrame is now ready for use in our visualisation.

In [ ]:
# Export the final DataFrame
df.to_csv("Dataset.csv", index=False)

In [16]:
# Export demo dataset (last 5000 rows)
demo_df = df.tail(5000)
demo_df.to_csv("Demo Dataset.csv", index=False)

In [18]:
# Final check
print(df.tail())
print("Final dataset shape:", df.shape)
print("Demo dataset shape:", demo_df.shape)
list(df.columns)

               id            slug            name   name_original  \
159458  1002225.0  dinosaur-world  Dinosaur World  Dinosaur World   
159459  1002221.0   stress-ball-2     Stress Ball     Stress Ball   
159460  1002231.0  astral-destiny  Astral Destiny  Astral Destiny   
159461  1002235.0   forgot-to-poo   Forgot To POO   Forgot To POO   
159462  1002233.0       blockhole      BLOCK+HOLE      BLOCK+HOLE   

                                              description  metacritic  \
159458  <p>As an explorer, you will come to an unknown...         NaN   
159459  <h3>Stress Ball: Confront the Stress of Jumpin...         NaN   
159460  <p><strong>Earth. The "S.T.A.R." (Space and Ti...         NaN   
159461  <p><strong>A casual, quirky platformer about a...         NaN   
159462  <p>Welcome to the world of classic puzzles!</p...         NaN   

       metacritic_platforms   released    tba              updated  ...  \
159458                   [] 2025-04-26  False  2025-04-28T10:47:47  ...

['id',
 'slug',
 'name',
 'name_original',
 'description',
 'metacritic',
 'metacritic_platforms',
 'released',
 'tba',
 'updated',
 'background_image',
 'background_image_additional',
 'website',
 'rating',
 'rating_top',
 'added',
 'playtime',
 'screenshots_count',
 'movies_count',
 'creators_count',
 'achievements_count',
 'parent_achievements_count',
 'reddit_url',
 'reddit_name',
 'reddit_description',
 'reddit_logo',
 'reddit_count',
 'twitch_count',
 'youtube_count',
 'reviews_text_count',
 'ratings_count',
 'suggestions_count',
 'alternative_names',
 'metacritic_url',
 'parents_count',
 'additions_count',
 'game_series_count',
 'reviews_count',
 'parent_platforms',
 'platforms',
 'stores',
 'developers',
 'genres',
 'tags',
 'publishers',
 'description_raw',
 'added_by_status.yet',
 'added_by_status.owned',
 'added_by_status.beaten',
 'added_by_status.toplay',
 'added_by_status.dropped',
 'added_by_status.playing',
 'reactions.4',
 'reactions.6',
 'reactions.7',
 'reactions.11'